## Data Ingestion

In this step, we load the raw dataset used for BFRB detection from the Helios wearable device.

The dataset consists of multimodal time-series sensor data, including:
- IMU (acceleration and rotation)
- Thermopile (temperature)
- Time-of-Flight (TOF) sensors

We read the raw training dataset (`train.csv`) into a pandas DataFrame for further processing.

This step verifies:
- The dataset can be successfully loaded
- The structure and dimensions of the data
- A preview of the data for initial inspection

The loaded dataset will be used in subsequent steps including:
- Exploratory Data Analysis (EDA)
- Data preprocessing and cleaning
- Feature extraction and model training

In [14]:
import pandas as pd

df = pd.read_csv(r"E:\CAPSTONE\2026-winter-capstone-project-2026winter-capstone-group-11\data\raw\train.csv")

print("Shape:", df.shape)
df.head()

Shape: (574945, 341)


,row_id,sequence_type,sequence_id,sequence_counter,subject,orientation,behavior,phase,gesture,acc_x,...,tof_5_v54,tof_5_v55,tof_5_v56,tof_5_v57,tof_5_v58,tof_5_v59,tof_5_v60,tof_5_v61,tof_5_v62,tof_5_v63
0,SEQ_000007_000000,Target,SEQ_000007,0,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,6.683594,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
1,SEQ_000007_000001,Target,SEQ_000007,1,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,6.949219,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
2,SEQ_000007_000002,Target,SEQ_000007,2,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,5.722656,...,-1.0,-1.0,112.0,119.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
3,SEQ_000007_000003,Target,SEQ_000007,3,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,6.601562,...,-1.0,-1.0,101.0,111.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
4,SEQ_000007_000004,Target,SEQ_000007,4,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,5.566406,...,-1.0,-1.0,101.0,109.0,125.0,-1.0,-1.0,-1.0,-1.0,-1.0


## Feature and Label Overview

We inspect the dataset structure by:
- Listing all feature columns
- Viewing unique gesture labels
- Counting the number of gesture classes

This helps understand the input features and classification setup.

In [15]:
print(df.columns.tolist())

print(df["gesture"].unique())
print("Num gestures:", len(df["gesture"].unique()))

['row_id', 'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'acc_x', 'acc_y', 'acc_z', 'rot_w', 'rot_x', 'rot_y', 'rot_z', 'thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5', 'tof_1_v0', 'tof_1_v1', 'tof_1_v2', 'tof_1_v3', 'tof_1_v4', 'tof_1_v5', 'tof_1_v6', 'tof_1_v7', 'tof_1_v8', 'tof_1_v9', 'tof_1_v10', 'tof_1_v11', 'tof_1_v12', 'tof_1_v13', 'tof_1_v14', 'tof_1_v15', 'tof_1_v16', 'tof_1_v17', 'tof_1_v18', 'tof_1_v19', 'tof_1_v20', 'tof_1_v21', 'tof_1_v22', 'tof_1_v23', 'tof_1_v24', 'tof_1_v25', 'tof_1_v26', 'tof_1_v27', 'tof_1_v28', 'tof_1_v29', 'tof_1_v30', 'tof_1_v31', 'tof_1_v32', 'tof_1_v33', 'tof_1_v34', 'tof_1_v35', 'tof_1_v36', 'tof_1_v37', 'tof_1_v38', 'tof_1_v39', 'tof_1_v40', 'tof_1_v41', 'tof_1_v42', 'tof_1_v43', 'tof_1_v44', 'tof_1_v45', 'tof_1_v46', 'tof_1_v47', 'tof_1_v48', 'tof_1_v49', 'tof_1_v50', 'tof_1_v51', 'tof_1_v52', 'tof_1_v53', 'tof_1_v54', 'tof_1_v55', 'tof_1_v56', 'tof_1_v57', 'tof_1_v58', 'tof_1_v59', '

## Label Construction

We convert the multi-class gesture labels into a binary classification task.

- BFRB gestures are labeled as 1
- Non-BFRB gestures are labeled as 0

This transformation enables binary classification and evaluation using F1-score.    

In [16]:
bfrb_gestures = {
    "Cheek - pinch skin",
    "Forehead - pull hairline",
    "Neck - scratch",
    "Neck - pinch skin",
    "Eyelash - pull hair",
    "Eyebrow - pull hair",
    "Forehead - scratch",
    "Above ear - pull hair"
}

df["label"] = df["gesture"].apply(lambda x: 1 if x in bfrb_gestures else 0)

print(df["label"].value_counts())

1    344058
0    230887
Name: label, dtype: int64


## Feature Selection

We select relevant sensor features from the dataset:

- IMU features: acceleration (`acc`) and rotation (`rot`)
- Thermopile features: temperature (`thm`)
- TOF features: a subset of proximity sensors (`tof`)

These selected features are combined to form the input feature set for modeling.    

In [17]:
imu_cols = [col for col in df.columns if "acc" in col or "rot" in col]
thm_cols = [col for col in df.columns if "thm" in col]

tof_cols = [col for col in df.columns if "tof" in col][:32]

feature_cols = imu_cols + thm_cols + tof_cols


print("IMU cols:", imu_cols)
print("THM cols:", thm_cols)
print("Feature count:", len(feature_cols))

IMU cols: ['acc_x', 'acc_y', 'acc_z', 'rot_w', 'rot_x', 'rot_y', 'rot_z']
THM cols: ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
Feature count: 44


## Data Cleaning

We remove rows with missing values in the selected feature columns.

This ensures:
- Clean input data for modeling
- No NaN values in sensor features
- Improved data quality and consistency

In [18]:
print("Before dropna:", df.shape)

df = df.dropna(subset=feature_cols).copy()

print("After dropna:", df.shape)
print("Remaining NaN count:", df[feature_cols].isna().sum().sum())

Before dropna: (574945, 342)
After dropna: (536305, 342)
Remaining NaN count: 0


## Sequence Construction

We group data by `sequence_id` to form time-series sequences.

- Each sequence represents a gesture instance
- Sequences with very short length are removed
- Labels are assigned per sequence

This prepares the data for sequence-based modeling.

In [19]:
sequences = []
labels = []

for seq_id, group in df.groupby("sequence_id"):
    seq = group[feature_cols].values
    
    if len(seq) < 10:
        continue
    
    sequences.append(seq)
    labels.append(group["label"].iloc[0])

print("Num sequences:", len(sequences))

Num sequences: 7595


## Sequence Padding

We pad all sequences to the same length for model input.

- Shorter sequences are padded with zeros
- Data is converted into a 3D array (samples × time × features)
- Labels are stored as a NumPy array

This ensures consistent input shape for training models.

In [20]:
import numpy as np

max_len = max(len(seq) for seq in sequences)

X = np.zeros((len(sequences), max_len, len(feature_cols)))

for i, seq in enumerate(sequences):
    X[i, :len(seq), :] = seq

y = np.array(labels)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Any NaN in X before FFT:", np.isnan(X).any())

X shape: (7595, 700, 44)
y shape: (7595,)
Any NaN in X before FFT: False


## FFT Transformation

We apply Fast Fourier Transform (FFT) along the time dimension.

- Transforms time-series data into frequency-domain features
- Uses magnitude values as model input
- Captures underlying signal patterns

This step prepares features for model training.

In [21]:
from scipy.fft import fft

X = np.abs(fft(X, axis=1))

print("Any NaN after FFT:", np.isnan(X).any())
print("FFT sample:", X[0, 0, :5])

Any NaN after FFT: False
FFT sample: [350.7265625  223.1875     317.93359375  15.02374268  16.0065918 ]


## Feature Normalization

We normalize the input features using standardization.

- Subtract mean and divide by standard deviation
- Applied across all samples and time steps
- Prevents scale differences between features

This improves model training stability.

In [22]:
mean = X.mean(axis=(0,1), keepdims=True)
std = X.std(axis=(0,1), keepdims=True) + 1e-8

X = (X - mean) / std

print("Mean:", X.mean())
print("Std:", X.std())
print("Any NaN:", np.isnan(X).any())

Mean: 1.4011280918743373e-14
Std: 0.9999999997266708
Any NaN: False


## Sequence Truncation

We truncate all sequences to a fixed maximum length.

- Keeps only the first 200 time steps
- Reduces memory usage and computation cost
- Ensures consistent input size for modeling

In [23]:
X = X[:, :200, :]
print("After truncate:", X.shape)

After truncate: (7595, 200, 44)


## Train-Test Split

We split the dataset into training and testing sets.

- 80% for training, 20% for testing
- Stratified split to preserve class distribution
- Fixed random seed for reproducibility

This enables proper evaluation of model performance.

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (6076, 200, 44)
Test: (1519, 200, 44)


## Tensor Conversion

We convert the data into PyTorch tensors.

- Features are converted to float tensors
- Labels are converted to integer tensors
- Prepares data for PyTorch model training

In [25]:
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

print(X_train.shape, y_train.shape)

torch.Size([6076, 200, 44]) torch.Size([6076])


## Model Setup

We import PyTorch and neural network modules.

These libraries are used to define and train deep learning models.

In [26]:
import torch
import torch.nn as nn

## GRU Model

We define a GRU-based neural network for sequence classification.

- Uses a 2-layer GRU to capture temporal dependencies
- Hidden representation is passed to a fully connected layer
- Outputs class predictions for binary classification

In [27]:
class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True, num_layers=2)
        self.fc = nn.Linear(hidden_dim, 2)
    
    def forward(self, x):
        _, h = self.gru(x)     # h: (1, batch, hidden)
        h = h[-1]       # (batch, hidden)
        return self.fc(h)

## Model Initialization

We initialize the GRU model, loss function, and optimizer.

- CrossEntropyLoss is used for classification
- Adam optimizer is used for training
- Learning rate is set to 0.0003

In [28]:
model = GRUModel(X_train.shape[2])

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

print("Input dim:", X_train.shape[1] * X_train.shape[2])

Input dim: 8800


## Model Training

We train the GRU model using mini-batch gradient descent.

- Data is processed in batches
- Loss is computed using CrossEntropyLoss
- Model parameters are updated using Adam optimizer
- Training runs for 20 epochs

This step learns patterns from the training data.

In [29]:
epochs = 20
batch_size = 128

for epoch in range(epochs):
    model.train()
    
    total_loss = 0
    
    for i in range(0, len(X_train), batch_size):
        X_batch = X_train[i:i+batch_size]
        y_batch = y_train[i:i+batch_size]
        
        optimizer.zero_grad()
        
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 31.4039
Epoch 2, Loss: 29.5590
Epoch 3, Loss: 27.7100
Epoch 4, Loss: 25.9054
Epoch 5, Loss: 24.9664
Epoch 6, Loss: 24.3381
Epoch 7, Loss: 24.0173
Epoch 8, Loss: 23.7877
Epoch 9, Loss: 23.5750
Epoch 10, Loss: 23.3844
Epoch 11, Loss: 23.2016
Epoch 12, Loss: 23.0131
Epoch 13, Loss: 22.7170
Epoch 14, Loss: 22.5148
Epoch 15, Loss: 22.5108
Epoch 16, Loss: 22.1662
Epoch 17, Loss: 22.2750
Epoch 18, Loss: 21.5886
Epoch 19, Loss: 21.1808
Epoch 20, Loss: 20.8239


## Model Evaluation

We evaluate the trained model using Binary F1-score.

- Convert model outputs to probabilities
- Apply a threshold to obtain predictions
- Compute F1-score on the test set

This measures classification performance.

In [ ]:
from sklearn.metrics import f1_score

model.eval()

with torch.no_grad():
    outputs = model(X_test)
    probs = torch.softmax(outputs, dim=1)[:, 1]

threshold = 0.3
preds = (probs > threshold).long()

f1 = f1_score(y_test.numpy(), preds.numpy())

print("Final Binary F1:", f1)

Final Binary F1: 0.8448355720240851


## Class Distribution

We examine the distribution of labels in the training set.

- Counts the number of samples per class
- Helps identify class imbalance

This is useful for understanding model performance.

In [ ]:
from collections import Counter
print(Counter(y_train.numpy()))

Counter({1: 3814, 0: 2262})
